In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI  # to talk with LLM
from langchain_core.documents import Document # to prepare text document
from langchain_text_splitters import RecursiveCharacterTextSplitter # to convert document to chunks
from langchain_openai import OpenAIEmbeddings # To convert chunks to vectors
from langchain_community.vectorstores import Chroma # to store vectors to Chromadb


C:\Users\Vamsi Krishna\AppData\Local\Temp\ipykernel_45836\1289032725.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma # to store vectors to Chromadb


In [2]:
llm=ChatOpenAI(model="gpt-5-nano",temperature=0)


### step 1 - Extracting text from pdf

In [9]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path="./pyspark.pdf"

loader=PyPDFLoader(pdf_path)

docs=loader.load()
docs

[Document(metadata={'producer': 'pdfTeX-1.40.14', 'creator': 'LaTeX with hyperref package', 'creationdate': '2019-07-21T12:17:14-05:00', 'author': 'Wenqiang Feng', 'title': 'Learning Apache Spark with Python', 'subject': '', 'keywords': '', 'moddate': '2019-07-21T12:17:14-05:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.1415926-2.5-1.40.14 (TeX Live 2013/Debian) kpathsea version 6.1.1', 'source': './pyspark.pdf', 'total_pages': 427, 'page': 0, 'page_label': '1'}, page_content='Learning Apache Spark with Python\nWenqiang Feng\nJuly 21, 2019'),
 Document(metadata={'producer': 'pdfTeX-1.40.14', 'creator': 'LaTeX with hyperref package', 'creationdate': '2019-07-21T12:17:14-05:00', 'author': 'Wenqiang Feng', 'title': 'Learning Apache Spark with Python', 'subject': '', 'keywords': '', 'moddate': '2019-07-21T12:17:14-05:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.1415926-2.5-1.40.14 (TeX Live 2013/Debian) kpathsea version 6.1.1', 'source':

#### STEP 2 : Create Chunks

In [10]:
##Split the document to chunks

splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=100)

chunks=splitter.split_documents(docs)

In [12]:
len(chunks)

781

In [ ]:
### Since we got 781 chunks which means 781 vectores will be created

### STEP 3 : Create Embedding Model

In [5]:
embedding_model=OpenAIEmbeddings(model="text-embedding-3-small")

### step 4 : Create Vector Store

In [ ]:
vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./Vector/"  ### this can be used if we can self the vector db instead of building in-memory vector db
)

### Step 5 : Semantic Search

In [17]:
vectorstore.similarity_search("What are internals of spark. give me a overview",k=20)

[Document(metadata={'total_pages': 427, 'page': 34, 'subject': '', 'trapped': '/False', 'page_label': '29', 'moddate': '2019-07-21T12:17:14-05:00', 'author': 'Wenqiang Feng', 'creator': 'LaTeX with hyperref package', 'producer': 'pdfTeX-1.40.14', 'creationdate': '2019-07-21T12:17:14-05:00', 'keywords': '', 'title': 'Learning Apache Spark with Python', 'source': './pyspark.pdf', 'ptex.fullbanner': 'This is pdfTeX, Version 3.1415926-2.5-1.40.14 (TeX Live 2013/Debian) kpathsea version 6.1.1'}, page_content='CHAPTER\nFOUR\nAN INTRODUCTION TO APACHE SPARK\nChinese proverb\nKnow yourself and know your enemy, and you will never be defeated– idiom, from Sunzi’s Art of War\n4.1 Core Concepts\nMost of the following content comes from [Kirillov2016]. So the copyright belongs to Anton Kirillov. I\nwill refer you to get more details from Apache Spark core concepts, architecture and internals.\nBefore diving deep into how Apache Spark works, lets understand the jargon of Apache Spark\n• Job: A piece

In [18]:
context=vectorstore.similarity_search("What are internals of spark. give me a overview",k=20)

In [19]:
#### Talk to LLM
response = llm.invoke(f"what are internals of spark. you can use the following context {context}")
print(response.content)

Here’s a concise overview of Apache Spark’s internals, based on the core concepts described in the material you provided.

High-level internal architecture
- Driver program
  - Runs on the client (master in effect) and coordinates the whole job.
  - Contains essential components like SparkContext, which represents the connection to the cluster and is used to create RDDs, accumulators, and broadcast variables.
  - DAGScheduler and TaskScheduler live here to orchestrate execution.
- Cluster manager
  - Manages resources across the cluster and can be Mesos, YARN, Spark Standalone, or local mode.
  - The driver negotiates with the cluster manager to acquire resources for executors.
- Executors
  - Run tasks scheduled by the driver.
  - Store computed data and results in memory, on disk, or off-heap.
  - Interact with storage systems as needed (e.g., HDFS, local storage).
- BlockManager
  - Provides interfaces for putting and retrieving data blocks both locally and remotely across various s

In [3]:
#### since we have pyisical vector db we dont have to create in-memory vector db . 
# we can directly connect to created vector db in Vector folder


#### Reuse the vector database

In [6]:
vectorstore_persist = Chroma( persist_directory="./Vector",
                                embedding_function=embedding_model)

C:\Users\Vamsi Krishna\AppData\Local\Temp\ipykernel_45836\2414475936.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore_persist = Chroma( persist_directory="./Vector",


In [7]:
vectorstore_persist.similarity_search("What are internals of spark. give me a overview",k=20)

[Document(metadata={'source': './pyspark.pdf', 'moddate': '2019-07-21T12:17:14-05:00', 'page_label': '29', 'total_pages': 427, 'producer': 'pdfTeX-1.40.14', 'creator': 'LaTeX with hyperref package', 'title': 'Learning Apache Spark with Python', 'keywords': '', 'creationdate': '2019-07-21T12:17:14-05:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.1415926-2.5-1.40.14 (TeX Live 2013/Debian) kpathsea version 6.1.1', 'trapped': '/False', 'subject': '', 'page': 34, 'author': 'Wenqiang Feng'}, page_content='CHAPTER\nFOUR\nAN INTRODUCTION TO APACHE SPARK\nChinese proverb\nKnow yourself and know your enemy, and you will never be defeated– idiom, from Sunzi’s Art of War\n4.1 Core Concepts\nMost of the following content comes from [Kirillov2016]. So the copyright belongs to Anton Kirillov. I\nwill refer you to get more details from Apache Spark core concepts, architecture and internals.\nBefore diving deep into how Apache Spark works, lets understand the jargon of Apache Spark\n• Job: A piece

In [8]:
context=vectorstore_persist.similarity_search("What are internals of spark. give me a overview",k=20)

In [9]:
#### Talk to LLM
response = llm.invoke(f"what are internals of spark. you can use the following context {context}")
print(response.content)

Here’s a concise summary of Spark internals based on the provided material. It covers the main components, architecture, and how Spark executes work.

Key components (driver vs. cluster)
- Spark Driver
  - The orchestrator that coordinates the entire application.
  - Contains core pieces like SparkContext, DAGScheduler, and TaskScheduler (responsible for translating user code into actual jobs and scheduling them).
- Executors
  - Run the tasks scheduled by the driver.
  - Store computed results in memory, on disk, or off-heap; interact with storage systems.
- Cluster Manager
  - Manages resources and allocation across the cluster (examples: Mesos, YARN, Spark Standalone, or local mode).
- BlockManager
  - Handles storage of data blocks across memory, disk, and off-heap; provides interfaces to push/pull blocks to/from local and remote stores.

Spark internals and how the work is organized
- Jobs, Stages, Tasks, and DAGs
  - A Job is a piece of user code that reads input, performs comput

In [ ]:
print("RAG Implementation Done")